# MERRA-2 — Surface Soil Wetness (GWETTOP)

Visualize the consolidated MERRA-2 `GWETTOP` (surface soil wetness, 0–0.05 m)
dataset from NASA GES DISC (M2TMNXLND v5.12.4).

- **Variable:** `GWETTOP` — surface soil wetness (fraction of saturation, 0–1)
- **Resolution:** 0.5° × 0.625° global grid
- **Period:** 1980–present (monthly)
- **Additional variables:** `GWETROOT` (root zone, 0–1 m), `GWETPROF` (full profile, variable depth)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

# --- Configuration ---
RUN_DIR = Path.home() / "data" / "nhf-runs" / "2026-03-17_gfv11_v0.1.0"
NC_PATH = RUN_DIR / "data" / "raw" / "merra2" / "merra2_consolidated.nc"
PRIMARY_VAR = "GWETTOP"


In [ ]:
ds = xr.open_dataset(NC_PATH)
print(ds)
print(f"\nTime range: {ds.time.values[0]} to {ds.time.values[-1]}")
print(f"Time steps: {ds.time.size}")
print(f"Grid: {ds.lat.size} lat x {ds.lon.size} lon")
print(f"Data variables: {list(ds.data_vars)}")
print(f"Conventions: {ds.attrs.get('Conventions', 'N/A')}")
print(f"Grid mapping: {ds[PRIMARY_VAR].attrs.get('grid_mapping', 'N/A')}")


## Single-month soil wetness map (global)

In [ ]:
TARGET_TIME = "2000-01-15"

da = ds[PRIMARY_VAR].sel(time=TARGET_TIME, method="nearest")
actual_time = str(da.time.values)[:10]

fig, ax = plt.subplots(figsize=(16, 6))
da.plot(ax=ax, cmap="YlGnBu", vmin=0, vmax=1, cbar_kwargs={"label": "fraction of saturation"})
ax.set_title(f"MERRA-2 Surface Soil Wetness (GWETTOP, 0–0.05 m) — {actual_time}")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

print(f"Stats for {actual_time}:")
print(f"  min:  {float(da.min()):.4f}")
print(f"  max:  {float(da.max()):.4f}")
print(f"  mean: {float(da.mean()):.4f}")
print(f"  NaN%: {float(da.isnull().mean()) * 100:.1f}%")


## All three soil wetness variables — single timestep comparison

In [ ]:
soil_vars = {
    "GWETTOP": "Surface (0–0.05 m)",
    "GWETROOT": "Root zone (0–1.0 m)",
    "GWETPROF": "Full profile (to bedrock)",
}

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for ax, (var, label) in zip(axes, soil_vars.items()):
    if var not in ds.data_vars:
        ax.set_title(f"{label}\n(not in dataset)")
        ax.set_visible(False)
        continue
    da = ds[var].sel(time=TARGET_TIME, method="nearest")
    da.plot(ax=ax, cmap="YlGnBu", vmin=0, vmax=1, cbar_kwargs={"label": "fraction"})
    ax.set_title(f"{var}\n{label}")
    ax.set_aspect("equal")

fig.suptitle(f"MERRA-2 Soil Wetness Variables — {actual_time}", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


## CONUS subset — calibration period mean (2000–2009)

In [ ]:
conus = ds[PRIMARY_VAR].sel(
    lat=slice(23.9, 50.1),
    lon=slice(-125.1, -65.9),
    time=slice("2000-01-01", "2009-12-31"),
)

mean_wetness = conus.mean(dim="time")

fig, ax = plt.subplots(figsize=(14, 8))
mean_wetness.plot(ax=ax, cmap="YlGnBu", vmin=0, vmax=1, cbar_kwargs={"label": "fraction of saturation"})
ax.set_title("Mean Surface Soil Wetness (GWETTOP) — CONUS 2000–2009")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

print(f"CONUS 2000-2009 mean stats:")
print(f"  min:  {float(mean_wetness.min()):.4f}")
print(f"  max:  {float(mean_wetness.max()):.4f}")
print(f"  mean: {float(mean_wetness.mean()):.4f}")


## Seasonal comparison (CONUS, 2000–2009)

In [ ]:
seasons = {
    "DJF (Winter)": conus.sel(time=conus.time.dt.month.isin([12, 1, 2])),
    "MAM (Spring)": conus.sel(time=conus.time.dt.month.isin([3, 4, 5])),
    "JJA (Summer)": conus.sel(time=conus.time.dt.month.isin([6, 7, 8])),
    "SON (Fall)": conus.sel(time=conus.time.dt.month.isin([9, 10, 11])),
}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, (label, seasonal_data) in zip(axes.flatten(), seasons.items()):
    seasonal_mean = seasonal_data.mean(dim="time")
    seasonal_mean.plot(ax=ax, cmap="YlGnBu", vmin=0, vmax=1, cbar_kwargs={"label": "fraction"})
    ax.set_title(f"{label} mean")
    ax.set_aspect("equal")

fig.suptitle("Seasonal Mean Surface Soil Wetness — CONUS 2000–2009", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


## Monthly time series (CONUS spatial mean)

In [ ]:
conus_full = ds[PRIMARY_VAR].sel(
    lat=slice(23.9, 50.1),
    lon=slice(-125.1, -65.9),
)
monthly_mean = conus_full.mean(dim=["lat", "lon"])

fig, ax = plt.subplots(figsize=(16, 4))
monthly_mean.plot(ax=ax, color="#2980b9", linewidth=0.5)
ax.set_ylabel("Mean Soil Wetness (fraction)")
ax.set_title("CONUS-mean monthly surface soil wetness (GWETTOP)")
ax.axvspan("2000-01-01", "2009-12-31", alpha=0.15, color="orange", label="Calibration period")
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.show()


## Monthly climatology (CONUS, 2000–2009)

In [ ]:
monthly_clim = conus.groupby("time.month").mean(dim=["time", "lat", "lon"])

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(monthly_clim.month, monthly_clim.values, color="#2980b9", edgecolor="white")
ax.set_xlabel("Month")
ax.set_ylabel("Mean Soil Wetness (fraction)")
ax.set_title("Monthly Climatology — CONUS 2000–2009 (GWETTOP)")
ax.set_xticks(range(1, 13))
ax.set_xticklabels(["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"])
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()


## CF compliance verification

In [ ]:
print("CF Compliance Checks:")
print(f"  Conventions attr:  {ds.attrs.get('Conventions', 'MISSING')}")
print(f"  Time dtype:        {ds.time.dtype}")
print(f"  Time first:        {ds.time.values[0]}")
print(f"  Time last:         {ds.time.values[-1]}")
print(f"  CRS variable:      {'crs' in ds.data_vars}")
if "crs" in ds.data_vars:
    print(f"  Grid mapping name: {ds['crs'].attrs.get('grid_mapping_name', 'MISSING')}")
    print(f"  CRS WKT:           {ds['crs'].attrs.get('crs_wkt', 'MISSING')[:60]}...")
print(f"  time_bnds:         {'time_bnds' in ds.data_vars}")
for var in ["GWETTOP", "GWETROOT", "GWETPROF"]:
    if var in ds.data_vars:
        print(f"  {var}:")
        print(f"    grid_mapping: {ds[var].attrs.get('grid_mapping', 'MISSING')}")
        print(f"    units:        {ds[var].attrs.get('units', 'MISSING')}")
        print(f"    long_name:    {ds[var].attrs.get('long_name', 'MISSING')}")


## Manifest provenance check

In [ ]:
import json

manifest_path = RUN_DIR / "manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    entry = manifest["sources"].get("merra2", {})
    print(f"Source key:       {entry.get('source_key', 'N/A')}")
    print(f"Access URL:       {entry.get('access_url', 'N/A')}")
    print(f"Period:           {entry.get('period', 'N/A')}")
    print(f"Variables:        {entry.get('variables', 'N/A')}")
    print(f"Consolidated NC:  {entry.get('consolidated_nc', 'N/A')}")
    print(f"Files downloaded: {len(entry.get('files', []))}")
    print(f"Downloaded UTC:   {entry.get('download_timestamp', 'N/A')}")
else:
    print(f"manifest.json not found at {manifest_path}")


In [ ]:
ds.close()
